# Gerçek Footstep referansı ile T-FOLEY

Referans: freefire66, “footsteps.wav”, Freesound 175954, CC0. Kaldırımda botla yürüyüş kaydının kamuya açık MP3 önizlemesi kullanılır; orijinal WAV değildir. Checksum doğrulandıktan sonra sabit 5–9 s bölümü mono 22050 Hz FLOAT WAV'a dönüştürülür. Peak normalizasyonu yapılmaz.

T-FOLEY referansın RMS/zaman yapısını alır; referans timbresini veya malzemesini birebir taklit etmez. `Footstep` sınıfı, seed 42, 100 sampling adımı. Bu tek çift, kalite veya controllability benchmark'ı değildir.


In [ ]:
from pathlib import Path
import sys, subprocess, json
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
from IPython.display import Audio, Markdown, display

ROOT = Path.cwd()
if not (ROOT / "src/baselines").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.audio.footstep_sample import prepare_footstep
from src.audio.tfoley_comparison import compare_pair

reference_path, source_metadata = prepare_footstep(ROOT)
source_metadata


## Referans kaydı

Üretim başlamadan kullanılacak gerçek kaydı dinleyebilirsiniz. Bu ilk player tek dosya için normalize eder. Sonraki karşılaştırmada iki sese aynı playback gain uygulanır.


In [ ]:
display(Markdown("**Gerçek referans: botla kaldırımda yürüyüş, 5–9 s**"))
reference_preview, preview_sr = sf.read(reference_path)
display(Audio(reference_preview, rate=preview_sr, normalize=True))


## Model üretimi

Modelde serbest metin koşulu yoktur. Mevcut CUDA PyTorch ve `requirements-tfoley.txt` gerekir. `--prepare`, resmî kaynak/checkpoint henüz yoksa indirir. `auto`, kullanılabilir CUDA GPU'yu seçer. `local_real_footstep` çıktıları paylaşılan önceki CPU koşumundan ayrı kaydedilir.


In [ ]:
STEM = "local_real_footstep"
command = [sys.executable, str(ROOT / "src/baselines/tfoley_inference.py"),
           "--prepare", "--device", "auto", "--steps", "100", "--seed", "42",
           "--class-name", "Footstep", "--target-audio", str(reference_path),
           "--output-name", STEM]
subprocess.run(command, cwd=ROOT, check=True)


## Referans ve üretilen ses

İki ses için aynı gain kullanılır; dosyalar değiştirilmez. İki sesin ortak en yüksek peak değeri 1 olacak şekilde her ikisine aynı katsayı uygulanır. Bağımsız player normalizasyonu RMS farkını gizleyebileceğinden bu karşılaştırmada kapalıdır.


In [ ]:
folder = ROOT / "results/audio/tfoley"
reference, sr = sf.read(folder / f"{STEM}_reference.wav")
generated, generated_sr = sf.read(folder / f"{STEM}.wav")
assert sr == generated_sr
playback_divisor = max(float(np.max(np.abs(reference))), float(np.max(np.abs(generated))), np.finfo(float).eps)
for title, signal in [("Gerçek referans", reference), ("T-FOLEY çıktısı — Footstep", generated)]:
    display(Markdown(f"**{title}**"))
    display(Audio(signal / playback_divisor, rate=sr, normalize=False))
print({"common_playback_divisor": playback_divisor})


## RMS ve olay zamanı tanıları

Frame 512/hop 128, aynı elliptic smoothing. MAE ve korelasyon enerji zarfını inceler; yüksek korelasyon ses karakteri benzerliğini kanıtlamaz. RMS tepe zamanları onset değildir. Relative prominence 15%, minimum mesafe 250 ms; en yakın eşleme aynı çıkış tepesini tekrar kullanabilir. Ek/eksik tepeler görsel kontrol gerektirir.


In [ ]:
metrics, fig = compare_pair(ROOT, STEM, source_metadata)
display(metrics)
plt.show()
run = json.loads((ROOT / f"experiments/baselines/tfoley/{STEM}_run.json").read_text(encoding="utf-8"))
run
